In [86]:
from math import factorial

from pyspark.sql.functions import (
    col, year, month, dayofmonth, weekofyear, date_format,
    weekday, when, expr, to_date, row_number
)
from pyspark.sql.window import Window
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
import ConnectionConfig as cc
cc.setupEnvironment()


Environment variables are set...


In [87]:
#config
cc.setupEnvironment()
print(cc.config.sections())

Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [88]:
spark = cc.startLocalCluster("FACT_TREASURE_FOUND",4)
spark.getActiveSession()

In [95]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [96]:
dateDimDf= spark.read.format("delta").load("./delta/DATE_DIM")
rainDimDf= spark.read.format("delta").load("./delta/RAIN_DIM")
seasonDimDf= spark.read.format("delta").load("./delta/SEASON_DIM")

In [120]:
fact_src = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option(
        "dbtable",
        "(select id, log_type, log_time, treasure_id,session_start from treasure_log) as subq"
    )
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
    .filter(col("log_type") == 2) # log_type 2 is 'Treasure Found'
    .withColumnRenamed("log_time", "LogDate")
    .withColumn("LogDate_date", to_date(col("LogDate")))
)
print("fact_src preview:")
fact_src.show(5)

fact_src preview:
+--------------------+--------+--------------------+--------------------+--------------------+------------+
|                  id|log_type|             LogDate|         treasure_id|       session_start|LogDate_date|
+--------------------+--------+--------------------+--------------------+--------------------+------------+
|[01 0F 33 3B F0 7...|       2|2022-07-22 23:34:...|[0E BE A3 5C 9B B...|2022-07-22 23:27:...|  2022-07-22|
|[01 0F 34 7F 4C D...|       2|2021-04-10 14:32:...|[C9 58 6D A2 53 E...|2021-04-10 14:24:...|  2021-04-10|
|[01 0F 36 78 8D 5...|       2|2022-06-11 21:21:...|[73 07 A1 06 B5 3...|2022-06-11 20:13:...|  2022-06-11|
|[01 0F 50 19 D3 8...|       2|2023-08-12 21:58:...|[12 59 1E F3 31 E...|2023-08-12 20:19:...|  2023-08-12|
|[01 0F 56 96 60 0...|       2|2022-05-23 20:01:...|[DB F7 CD 33 8C 2...|2022-05-23 18:29:...|  2022-05-23|
+--------------------+--------+--------------------+--------------------+--------------------+------------+
only showi

In [121]:

treasure_stages_df = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option("dbtable", "treasure_stages") # Direct de tabelnaam
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
)
print("treasure_stages_df preview:")
treasure_stages_df.show(5)
treasure_stages_df.createOrReplaceTempView("treasure_stages")

stage_df = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option("dbtable", "stage") # Direct de tabelnaam
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
)
print("stage_df preview:")
stage_df.show(5)
stage_df.createOrReplaceTempView("stage")


treasure_stages_df preview:
+--------------------+--------------------+
|         treasure_id|           stages_id|
+--------------------+--------------------+
|[00 00 3E 2C B1 4...|[62 EA 28 A8 51 3...|
|[00 00 3E 2C B1 4...|[A0 96 DC 61 F0 A...|
|[00 00 3E 2C B1 4...|[AA C9 B1 42 CB 8...|
|[00 00 3E 2C B1 4...|[FD 9F F6 5F EA 6...|
|[00 03 72 3C 7C C...|[1B 32 E8 9A A6 2...|
+--------------------+--------------------+
only showing top 5 rows
stage_df preview:
+--------------------+--------------+--------------------+------------------+------------------+---------------+----+----------+
|                  id|container_size|         description|          latitude|         longitude|sequence_number|type|visibility|
+--------------------+--------------+--------------------+------------------+------------------+---------------+----+----------+
|[00 8F 79 6A C9 2...|             1|In voluptate earu...|  78.2210727266575|15.637835959156016|              3|   0|         2|
|[00 8F 89 FB 21 0

In [122]:
# Vind de laatste stage (hoogste sequence_number) voor elke treasure_id
window_spec_stage = Window.partitionBy("treasure_id").orderBy(col("sequence_number").desc())
last_stage_per_treasure = (
    spark.sql("""
        SELECT
            ts.treasure_id,
            s.id as stage_id,
            s.sequence_number,
            s.latitude,
            s.longitude
        FROM
            treasure_stages ts
        JOIN
            stage s ON ts.stages_id = s.id
    """)
    .withColumn("rn", row_number().over(window_spec_stage))
    .filter(col("rn") == 1)
    .drop("rn", "sequence_number", "stage_id")
    .withColumnRenamed("city", "TreasureCity")
    .withColumnRenamed("country", "TreasureCountry")
)
print("last_stage_per_treasure preview:")
last_stage_per_treasure.show()
last_stage_per_treasure.createOrReplaceTempView("last_stage_per_treasure")

last_stage_per_treasure preview:
+--------------------+------------------+-------------------+
|         treasure_id|          latitude|          longitude|
+--------------------+------------------+-------------------+
|[00 03 72 3C 7C C...|12.568262445880535|  77.92589024707661|
|[00 04 62 1B 29 E...|51.728693518723624|-2.1977132499510175|
|[00 06 05 10 FC 0...| 34.06963357435847| -89.88196811630556|
|[00 08 A9 BF BE 4...| 22.77425290662822|  88.19301469524981|
|[00 08 B9 20 1B F...|27.994826260609962|  -82.2164858018488|
|[00 09 57 54 16 D...|11.168920300382176|  77.60795611973033|
|[00 0B 12 E3 93 2...|30.235295899656442|  78.14218071988124|
|[00 0E 3E 40 56 D...| 34.93535875366291|  -79.7625650335257|
|[00 0F 1D 4F FF A...|-3.803613250238623| -45.23241860685075|
|[00 0F 44 E2 4E D...|10.844275178123029|  78.77381030667783|
|[00 10 05 86 4E 3...| 34.16628716054675| -95.37296713235307|
|[00 11 6E F0 A5 0...| 63.13177486192953|  29.99343275870074|
|[00 11 E5 15 88 8...| 41.03316966236

In [123]:
# --- Start Fact Table constructie ---
fact_enriched = fact_src.join(
    last_stage_per_treasure,
    on="treasure_id",
    how="left"
).drop("log_type")
fact_enriched.show(5)


+--------------------+--------------------+--------------------+--------------------+------------+------------------+------------------+
|         treasure_id|                  id|             LogDate|       session_start|LogDate_date|          latitude|         longitude|
+--------------------+--------------------+--------------------+--------------------+------------+------------------+------------------+
|[C9 58 6D A2 53 E...|[01 0F 34 7F 4C D...|2021-04-10 14:32:...|2021-04-10 14:24:...|  2021-04-10| 8.857638522098133|124.78459567092725|
|[73 07 A1 06 B5 3...|[01 0F 36 78 8D 5...|2022-06-11 21:21:...|2022-06-11 20:13:...|  2022-06-11|51.247517376776166|19.053300329019624|
|[0E BE A3 5C 9B B...|[01 0F 33 3B F0 7...|2022-07-22 23:34:...|2022-07-22 23:27:...|  2022-07-22|11.890348647562076| 78.91736347382647|
|[DB F7 CD 33 8C 2...|[01 0F 56 96 60 0...|2022-05-23 20:01:...|2022-05-23 18:29:...|  2022-05-23|25.412879672476098| 74.10663621557221|
|[12 59 1E F3 31 E...|[01 0F 50 19 D3 8..

In [124]:
#  Eerst extraheer dag, week en jaar uit LogDate_date (tijd negeren)
fact_enriched_clean = (
    fact_enriched
    .withColumn("log_day", dayofmonth("LogDate_date"))
    .withColumn("log_week", weekofyear("LogDate_date"))
    .withColumn("log_year", year("LogDate_date"))
)

#  Join met DateDim op dag/week/jaar
fact_with_date_key = (
    fact_enriched_clean.join(
        dateDimDf.select("DateSurKey", "Day", "Week", "Year", "MonthOfTheYear"),
        (fact_enriched_clean["log_day"] == dateDimDf["Day"]) &
        (fact_enriched_clean["log_week"] == dateDimDf["Week"]) &
        (fact_enriched_clean["log_year"] == dateDimDf["Year"]),
        how="left"
    )
    .drop("log_day", "log_week", "log_year", "Day", "Week", "Year")
)

print("fact_with_date_key preview:")
fact_with_date_key.show()


fact_with_date_key preview:
+--------------------+--------------------+--------------------+--------------------+------------+------------------+-------------------+--------------------+--------------+
|         treasure_id|                  id|             LogDate|       session_start|LogDate_date|          latitude|          longitude|          DateSurKey|MonthOfTheYear|
+--------------------+--------------------+--------------------+--------------------+------------+------------------+-------------------+--------------------+--------------+
|[46 45 F7 25 53 9...|[01 0F D5 80 37 A...|2020-11-24 18:07:...|2020-11-24 16:55:...|  2020-11-24|-27.01046477858667| -49.22084259044665|d2104167-d936-4f8...|            11|
|[C9 58 6D A2 53 E...|[01 0F 34 7F 4C D...|2021-04-10 14:32:...|2021-04-10 14:24:...|  2021-04-10| 8.857638522098133| 124.78459567092725|5b3b7df7-c484-462...|             4|
|[16 D9 A6 CD 38 6...|[01 10 0A 3E F1 9...|2022-08-21 23:55:...|2022-08-21 23:36:...|  2022-08-21| 27.

In [125]:
# 2. SeasonDim koppelen
def get_season(month_of_year, latitude):
    if latitude is None or month_of_year is None: return "UNKNOWN"
    if latitude > 0: # Noordelijk halfrond
        if month_of_year >= 3 and month_of_year <= 5: return "Lente"
        elif month_of_year >= 6 and month_of_year <= 8: return "Zomer"
        elif month_of_year >= 9 and month_of_year <= 11: return "Herfst"
        else: return "Winter"
    elif latitude < 0: # Zuidelijk halfrond
        if month_of_year >= 9 and month_of_year <= 11: return "Lente"
        elif month_of_year >= 12 or month_of_year <= 2: return "Zomer"
        elif month_of_year >= 3 and month_of_year <= 5: return "Herfst"
        else: return "Winter"


In [126]:
get_season_udf = udf(get_season, StringType())

fact_with_season_key = fact_with_date_key.withColumn(
    "DeterminedSeason",
    get_season_udf(col("MonthOfTheYear"), col("latitude"))
).join(
    seasonDimDf.select("SeasonName", "SeasonSurKey"),
    col("DeterminedSeason") == seasonDimDf["SeasonName"],
    how="left"
).drop("DeterminedSeason", "SeasonName", "MonthOfTheYear")
print("fact_with_season_key preview:")
fact_with_season_key.show(5)

fact_with_season_key preview:
+--------------------+--------------------+--------------------+--------------------+------------+------------------+------------------+--------------------+--------------------+
|         treasure_id|                  id|             LogDate|       session_start|LogDate_date|          latitude|         longitude|          DateSurKey|        SeasonSurKey|
+--------------------+--------------------+--------------------+--------------------+------------+------------------+------------------+--------------------+--------------------+
|[C9 58 6D A2 53 E...|[01 0F 34 7F 4C D...|2021-04-10 14:32:...|2021-04-10 14:24:...|  2021-04-10| 8.857638522098133|124.78459567092725|5b3b7df7-c484-462...|4d9bb07a-f4a2-4f4...|
|[73 07 A1 06 B5 3...|[01 0F 36 78 8D 5...|2022-06-11 21:21:...|2022-06-11 20:13:...|  2022-06-11|51.247517376776166|19.053300329019624|70b100c0-256f-4a4...|ebb03bb0-7282-4d0...|
|[0E BE A3 5C 9B B...|[01 0F 33 3B F0 7...|2022-07-22 23:34:...|2022-07-22 

In [127]:
city_src = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option(
        "dbtable",
        "(select city_id, postal_code from city) as subq"
    )
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
)
print("city_src preview:")
city_src.show(15)

city_src preview:
+--------------------+-----------+
|             city_id|postal_code|
+--------------------+-----------+
|[00 00 0B 50 FA 3...|   513-1121|
|[00 00 3E 94 C1 D...|     678 01|
|[00 00 44 4E FB C...|       3133|
|[00 00 46 73 9F A...|      35320|
|[00 00 4A B0 6E 4...|   500-8844|
|[00 00 60 7D 16 1...|     152636|
|[00 00 7E 93 82 4...|     413528|
|[00 00 9A 97 B5 9...|      66072|
|[00 00 A2 6E C3 C...|   465-0058|
|[00 00 BC 55 B4 B...|      48200|
|[00 00 C3 10 D2 3...|      07046|
|[00 00 C6 BB 5E B...|   2690-563|
|[00 00 D8 F6 1D D...|      78570|
|[00 00 FA C7 77 E...|      27958|
|[00 01 40 AF 63 3...|        HS7|
+--------------------+-----------+
only showing top 15 rows


In [128]:
weather_history_df = spark.read.option("multiline", "true").json("./Weatherhistory/weerhistoriek.json")
weather_history_df.show(30, truncate=False)

+--------------------+----------------+--------------------+------------------------------------------------+----------+--------+
|city                |main            |timestamp           |weather                                         |wind      |zipCode |
+--------------------+----------------+--------------------+------------------------------------------------+----------+--------+
|Kishitacho          |{22.1, 55, 22.5}|2025-10-05T08:00:00Z|{clear sky, 01d, 800, Clear}                    |{150, 3.2}|513-1121|
|Kishitacho          |{25.0, 60, 24.8}|2025-10-05T12:00:00Z|{few clouds, 02d, 801, Clouds}                  |{190, 4.1}|513-1121|
|Kishitacho          |{20.0, 75, 20.2}|2025-10-05T16:00:00Z|{light rain, 10d, 500, Rain}                    |{220, 5.0}|513-1121|
|Blansko             |{13.2, 92, 13.5}|2025-10-05T07:00:00Z|{mist, 50d, 741, Mist}                          |{90, 1.8} |678 01  |
|Blansko             |{18.1, 70, 18.4}|2025-10-05T13:00:00Z|{broken clouds, 04d, 803, Clou

In [129]:
from pyspark.sql.functions import col

weather_flat = weather_history_df.select(
    col("zipCode"),
    col("weather.id").alias("weather_id")
)

#  alle city’s behouden, ook zonder weerdata
joined_df = (
    city_src.join(
        weather_flat,
        city_src["postal_code"] == weather_flat["zipCode"],
        "left"
    )
)

print("Resultaat:")
joined_df.show(40,truncate=False)


Resultaat:
+-------------------------------------------------+-----------+--------+----------+
|city_id                                          |postal_code|zipCode |weather_id|
+-------------------------------------------------+-----------+--------+----------+
|[00 00 0B 50 FA 30 4E 70 9A 68 5D DC 4D BB 0B 6D]|513-1121   |513-1121|500       |
|[00 00 0B 50 FA 30 4E 70 9A 68 5D DC 4D BB 0B 6D]|513-1121   |513-1121|801       |
|[00 00 0B 50 FA 30 4E 70 9A 68 5D DC 4D BB 0B 6D]|513-1121   |513-1121|800       |
|[00 00 3E 94 C1 D6 42 36 89 9D 33 E7 E6 F0 84 95]|678 01     |678 01  |800       |
|[00 00 3E 94 C1 D6 42 36 89 9D 33 E7 E6 F0 84 95]|678 01     |678 01  |803       |
|[00 00 3E 94 C1 D6 42 36 89 9D 33 E7 E6 F0 84 95]|678 01     |678 01  |741       |
|[00 00 44 4E FB C8 47 A0 84 97 59 3B C3 62 4E 65]|3133       |3133    |802       |
|[00 00 44 4E FB C8 47 A0 84 97 59 3B C3 62 4E 65]|3133       |3133    |211       |
|[00 00 44 4E FB C8 47 A0 84 97 59 3B C3 62 4E 65]|3133       |31

In [130]:
# WeatherSurKey toewijzen
fact_with_surkey = joined_df.withColumn(
    "WeatherSurKey",
    when((col("weather_id") >= 200) & (col("weather_id") <= 699), 1)
    .when(col("weather_id").isNotNull(), 2)
    .otherwise(3)
)


fact_with_surkey.select(
    "city_id", "postal_code", "weather_id", "WeatherSurKey"
).show(40,truncate=False)

+-------------------------------------------------+-----------+----------+-------------+
|city_id                                          |postal_code|weather_id|WeatherSurKey|
+-------------------------------------------------+-----------+----------+-------------+
|[00 00 0B 50 FA 30 4E 70 9A 68 5D DC 4D BB 0B 6D]|513-1121   |500       |1            |
|[00 00 0B 50 FA 30 4E 70 9A 68 5D DC 4D BB 0B 6D]|513-1121   |801       |2            |
|[00 00 0B 50 FA 30 4E 70 9A 68 5D DC 4D BB 0B 6D]|513-1121   |800       |2            |
|[00 00 3E 94 C1 D6 42 36 89 9D 33 E7 E6 F0 84 95]|678 01     |800       |2            |
|[00 00 3E 94 C1 D6 42 36 89 9D 33 E7 E6 F0 84 95]|678 01     |803       |2            |
|[00 00 3E 94 C1 D6 42 36 89 9D 33 E7 E6 F0 84 95]|678 01     |741       |2            |
|[00 00 44 4E FB C8 47 A0 84 97 59 3B C3 62 4E 65]|3133       |802       |2            |
|[00 00 44 4E FB C8 47 A0 84 97 59 3B C3 62 4E 65]|3133       |211       |1            |
|[00 00 44 4E FB C8 4

In [131]:
treasure_src = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option(
        "dbtable",
        "(select id, city_city_id from treasure) as subq"
    )
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
)
print("treasure_src preview:")
treasure_src.show(15)

treasure_src preview:
+--------------------+--------------------+
|                  id|        city_city_id|
+--------------------+--------------------+
|[00 00 3E 2C B1 4...|[62 34 F0 0E 0E 9...|
|[00 03 72 3C 7C C...|[59 07 42 E8 55 1...|
|[00 04 39 A0 98 7...|[DB 39 44 FB 1D 0...|
|[00 04 62 1B 29 E...|[66 97 B3 4B 1F 1...|
|[00 05 1E A2 05 8...|[38 EC 6D 3C 5A 1...|
|[00 05 A4 FF 38 0...|[99 F1 55 1E 80 8...|
|[00 06 05 10 FC 0...|[50 3E 91 AA F3 3...|
|[00 08 A9 BF BE 4...|[59 B1 A5 28 DD C...|
|[00 08 B8 9F B1 D...|[F6 A7 AD 78 3F 1...|
|[00 08 B9 20 1B F...|[27 9E 9E 63 9E A...|
|[00 08 E2 70 CB A...|[AE A0 A3 09 22 1...|
|[00 09 57 54 16 D...|[B2 BE 60 8E A5 B...|
|[00 09 5D DB 3B C...|[79 FC E5 64 B9 B...|
|[00 0A 1E 88 C1 1...|[2F 08 AB 1D 02 B...|
|[00 0A 91 0B 0E 3...|[63 58 42 A3 F6 5...|
+--------------------+--------------------+
only showing top 15 rows


In [135]:
from pyspark.sql.functions import col

fact_linked = (
    fact_with_season_key
    .join(treasure_src, fact_with_season_key["treasure_id"] == treasure_src["id"], "left")
    .join(
        fact_with_surkey.select("city_id", "WeatherSurKey"),
        treasure_src["city_city_id"] == fact_with_surkey["city_id"],
        "left"
    )
    .drop(treasure_src["id"])
    .drop(fact_with_surkey["city_id"])
)


print("fact_linked preview (met Season + Weather):")
fact_linked.select(
    "DateSurKey",
    "SeasonSurKey",
    "WeatherSurKey"
).show(20, truncate=False)


fact_linked preview (met Season + Weather):
+------------------------------------+------------------------------------+-------------+
|DateSurKey                          |SeasonSurKey                        |WeatherSurKey|
+------------------------------------+------------------------------------+-------------+
|5b3b7df7-c484-4628-b6a2-dbf9a7bd7e5b|4d9bb07a-f4a2-4f41-9d8d-108d9c88f66c|3            |
|5e00d4ab-94a9-4d6c-a853-eb34b2839d3a|ebb03bb0-7282-4d00-910c-a850db3b8e57|3            |
|71b72559-1b67-4226-9c4d-206ca272418e|ebb03bb0-7282-4d00-910c-a850db3b8e57|3            |
|246ccc1b-07a5-40f6-a751-255176b6515a|ebb03bb0-7282-4d00-910c-a850db3b8e57|3            |
|eace24bb-afdb-44a0-811c-4810cf07f8de|617c7216-4bd8-41c9-a8ef-a3bc2083b4ff|3            |
|516fcaed-cc28-411d-abd2-0888d09ddc26|ebb03bb0-7282-4d00-910c-a850db3b8e57|3            |
|137159ca-0306-4afc-83ab-76137b9395f1|ebb03bb0-7282-4d00-910c-a850db3b8e57|3            |
|ffc64532-9a6f-4389-800e-416563a9ba9d|4d9bb07a-f4a2-4f41